<a href="https://colab.research.google.com/github/faridelya/Finetuning-Large-Language-Models/blob/main/3_ways_Fine_Tuning_LLAMA_2_with_autotrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3 Easy ways to Fine-tune LLAMA 2  
There are three major ways to fine tune open source LLMs like LLAMA
- autotrain
- QLora
- lamini


# Fine tuning using autotrain

In [ ]:
!pip install autotrain-advanced
!pip install huggingface_hub

In [ ]:
# update torch
!autotrain setup --update-torch

In [ ]:
# Login to huggingface

from huggingface_hub import notebook_login
notebook_login()

llm - the type of model you want to fine-tune
--project_name - the name you will like to give your project

--model - the name of the model you want to fine-tune

--data_path - path to the data that will be used in fine-tuning the model

--text_column - column name the contain the instruction and response or human and assitant

--use_peft - the efficient adapatin method the model should use. PEFT(Parameter-Efficient Fine-Tuning)

--use_int4 - the precison of the model

--learning_rate - speed of conversion for the learning process, the smaller the number, the slower the fine-tuning process

--train_batch_size - number of batch to divide you data into

--num_train_epochs - number of training epochs

--trainer - type of trainer to use. sft(supervised fine-tuning)

--model_max_length - content window for the model

--push_to_hub - push the fine-tuned model to huggingface

--repo_id - repo id on huggingface to push the model

--block_size  

> training.log - stores log generated during training



Notes

A sharded model divides the model to shards to help in memory efficieny if you dont have large amount of memory

data_path could be path to local storage or huggingface dataset path. You can also upload your dataset to huggingface. The dataset should be a .csv format and should be name "train.csv"

The csv file should contain a column with string format. The values of the columns should have ## Instruction and ## Response. The instruction a question that can be asked while the response is possible answer to the question. It could also also contain ## Input which is an addition information to the instruction.Another possible way is to use the key words ## Human and ## Assitant or ## Instruction, ##Input, ##Output


LoRA: LORA: Low-Rank Adaption of LLMs

In [ ]:
!autotrain llm --train --project_name "llama2-autotrain-openassitant" --model TinyPixel/Llama-2-7B-bf16-sharded --data_path timdettmers/openassistant-guanaco --text_column text --use_peft --use_int4 --learning_rate 0.4 --train_batch_size 3 --num_train_epochs 2 --trainer sft --model_max_length 1048 --push_to_hub --repo_id trojrobert/llama2-autotrain-openassistant --block_size 1048 > training.log

In [ ]:
!nvidia-smi

In [ ]:
!autotrain llm
--train
--project_name "llama2-autotrain-openassitant"
--model TinyPixel/Llama-2-7B-bf16-sharded
--data_path timdettmers/openassistant-guanaco
--text_column text
--use_peft
--use_int4
--learning_rate 0.4
--train_batch_size 3
--num_train_epochs 2
--trainer sft
--model_max_length 1048
--push_to_hub
--repo_id trojrobert/llama2-autotrain-openassistant
--block_size 1048 > training.log

In [ ]:
from llama import BasicModelRunner

model = BasicModelRunner("TinyPixel/Llama-2-7B-bf16-sharded")
model.load_data_from_jsonlines("lamini_docs.jsonl")

trainer = Trainer(
    model=base_model,
    model_flops=model_flops,
    total_steps=max_steps,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

training_output = trainer.train()

In [ ]:
from trl import SFTTrainer


trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
)